# 05 - Level-only baseline

Clusters the days with only two numbers per day: the mean and the standard deviation of that day's price (in logs for `real_price_log`). No scalograms, no CWT and no VAE. It is the same clustering input as notebooks 02 and 03 without the latent vector.

It answers the main question of the thesis: does the intraday shape of the price, which the CWT and the VAE capture, add information about the market regime beyond how expensive and how volatile the day was? A model only adds value if it beats this baseline in persistence (fewer switches per year) and in how well its most expensive regime captures the El Nino days of 2015-2016 and 2023-2024.

Pipeline: daily price level (mean, std) -> k-means or GMM -> regime

Runs are saved like any other run, in `experiments/level_only/<clustering>/`, with `study = "baseline"`, so notebook 04 can compare them with any model. Nothing is trained, so a full run takes about a minute (most of it is the t-SNE).

<d.benavidess@uniandes.edu.co>

In [1]:
import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import pandas as pd
import altair as alt
from sklearn.manifold import TSNE

alt.data_transformers.enable('vegafusion')

from src.data import load_context, chronological_split
from src.clustering import with_level, level_columns, select_k
from src.plots import selection_chart
from src.runs import prepare_runs, save_all, cluster_runs, k_sweep_table, tsne_grid, regime_strips
from src import experiments as exp

In [2]:
import warnings
warnings.filterwarnings("ignore")

## 0. Configuration

`scalogram_set` only decides which daily level is used: `"real_price_log"` (log_price_mean, log_price_std) or `"real_price"` (price_mean, price_std in COP/kWh). The scalogram images are not read.

In [3]:
scalogram_set = "real_price_log"
clustering = "kmeans"
k_values = [2, 3, 4, 5, 6]
study = "baseline"

config = {
    "model": "level_only",
    "study": study,
    "scalogram_set": scalogram_set,
    "clustering": clustering,
    "k_values": k_values,
    "k_range": [2, 3, 4, 5, 6, 7, 8],
    "features": "daily level (mean, std)",
    "train_ratio": 0.6,
    "val_ratio": 0.2,
    "seed": 42,
}

## 1. Run folders

In [4]:
config, run_dirs, run_configs, _ = prepare_runs(config)
for k, run_dir in run_dirs.items():
    print(f"k={k}: {exp.run_id(run_dir)}")

k=2: level_only/kmeans/001_real_price_log_kmeans_k2
k=3: level_only/kmeans/002_real_price_log_kmeans_k3
k=4: level_only/kmeans/003_real_price_log_kmeans_k4
k=5: level_only/kmeans/004_real_price_log_kmeans_k5
k=6: level_only/kmeans/005_real_price_log_kmeans_k6


## 2. Data

In [5]:
scalogram_dir = Path("../data/processed/scalograms") / config["scalogram_set"]
data_dir = Path("../data/processed")

metadata = pd.read_csv(scalogram_dir / "scalogram_metadata.csv", parse_dates=["start_datetime", "end_datetime"])
context = load_context(data_dir, metadata)
train_idx, val_idx, test_idx = chronological_split(len(metadata), config["train_ratio"], config["val_ratio"])
split_dates = [context["date"].iloc[val_idx[0]], context["date"].iloc[test_idx[0]]]

splits = exp.split_table(context, {"train": train_idx, "val": val_idx, "test": test_idx})
print(splits)
save_all(run_dirs, exp.save_table, "splits", splits, show=False)

features = with_level(np.zeros((len(metadata), 0)), metadata, train_idx)
print(f"Level columns: {level_columns(metadata)}, z-scored on training days")
print(f"Features: {features.shape}, max z-score {features.max():.1f}")

+-------+------+------------+------------+
| Split | Days |    From    |     To     |
+-------+------+------------+------------+
| train | 5825 | 2000-01-01 | 2015-12-12 |
|  val  | 1942 | 2015-12-13 | 2021-04-06 |
|  test | 1942 | 2021-04-07 | 2026-07-31 |
+-------+------+------------+------------+
Level columns: ('log_price_mean', 'log_price_std'), z-scored on training days
Features: (9709, 2), max z-score 6.1


## 3. Choice of k

Same table as notebooks 02 and 03, on the two level columns only.

In [6]:
selection = select_k(features, train_idx, config["clustering"], config["k_range"], config["seed"])
selection_table = exp.selection_table(selection)
print(selection_table)
save_all(run_dirs, exp.save_table, "k_selection", selection_table, show=False)
selection_chart(selection, config["k_values"][0])

+---+---------+------------+----------------+-------------------+
| k | Inertia | Silhouette | Davies-Bouldin | Calinski-Harabasz |
+---+---------+------------+----------------+-------------------+
| 2 |  7,053  |   0.417    |     0.968      |       3,795       |
| 3 |  4,634  |   0.393    |     0.873      |       4,407       |
| 4 |  3,717  |   0.370    |     0.885      |       4,140       |
| 5 |  2,973  |   0.366    |     0.883      |       4,247       |
| 6 |  2,444  |   0.372    |     0.803      |       4,385       |
| 7 |  2,109  |   0.347    |     0.837      |       4,388       |
| 8 |  1,834  |   0.371    |     0.793      |       4,446       |
+---+---------+------------+----------------+-------------------+


alt.HConcatChart(...)

## 4. One run per k

The t-SNE map is made on the same two level columns so the figure is comparable with the model runs. The feature checks add the 7-day mean of the level, to show how much of the day-to-day switching smoothing removes.

In [7]:
rng = np.random.default_rng(config["seed"])
plot_idx = np.sort(rng.choice(len(features), size=min(5000, len(features)), replace=False))
coords = TSNE(n_components=2, perplexity=30, random_state=config["seed"]).fit_transform(features[plot_idx])

level_cols = list(level_columns(metadata))
metadata_7d = metadata.copy()
metadata_7d[level_cols] = metadata[level_cols].rolling(7, min_periods=1).mean()
feature_sets = {
    "level only": features,
    "level 7d": with_level(np.zeros((len(metadata), 0)), metadata_7d, train_idx),
}
results = cluster_runs(run_dirs, run_configs, features, context, train_idx, split_dates, selection,
                       coords, plot_idx, feature_sets)

k=2: level_only/kmeans/001_real_price_log_kmeans_k2  silhouette=0.364  mean run=6.2 d  switches/yr=58.6
k=3: level_only/kmeans/002_real_price_log_kmeans_k3  silhouette=0.394  mean run=5.6 d  switches/yr=64.7
k=4: level_only/kmeans/003_real_price_log_kmeans_k4  silhouette=0.368  mean run=5.1 d  switches/yr=71.1
k=5: level_only/kmeans/004_real_price_log_kmeans_k5  silhouette=0.346  mean run=3.9 d  switches/yr=94.4
k=6: level_only/kmeans/005_real_price_log_kmeans_k6  silhouette=0.351  mean run=3.3 d  switches/yr=111.3


In [12]:
tsne_grid(coords, plot_idx, results)

alt.VConcatChart(...)

## 5. Comparison across k

Same columns as the k sweep of notebooks 02 and 03. Read the El Nino columns against `top_share_pct`.

In [9]:
k_sweep = k_sweep_table(results)
save_all(run_dirs, exp.save_table, "k_sweep", k_sweep, show=False)
k_sweep

,silhouette,davies_bouldin,calinski_harabasz,mean_run_days,switches_per_year,top_share_pct,top_price,top_reservoir,nino_2015_16_in_top_pct,nino_2023_24_in_top_pct
k,,,,,,,,,,
2,0.364,1.084,4693.608,6.228,58.612,70.769,358.550,0.657,95.551,84.776
3,0.394,0.832,7479.232,5.645,64.668,33.567,551.018,0.618,89.168,87.164
4,0.368,0.861,7669.784,5.132,71.139,11.155,919.141,0.596,42.747,77.313
5,0.346,0.895,7379.909,3.870,94.350,9.929,963.820,0.594,41.586,72.239
6,0.351,0.864,7282.510,3.280,111.317,8.394,1019.909,0.590,40.039,64.478


In [10]:
for k, r in results.items():
    print(f"k={k}  {exp.run_id(run_dirs[k])}")
    print(exp.regime_profile_table(r["profile"]))
    print("Share of each weekday in each regime (%)")
    print(r["weekday"].to_string(), end="\n\n")

k=2  level_only/kmeans/001_real_price_log_kmeans_k2
+--------+------+---------+------------+-----------+----------+-----------+
| Regime | Days | Share % | Real price | Reservoir | Mean ONI | El Nino % |
+--------+------+---------+------------+-----------+----------+-----------+
|   0    | 2838 |   29.2  |   213.2    |   0.730   |  -0.27   |    10.5   |
|   1    | 6871 |   70.8  |   358.6    |   0.657   |   0.11   |    30.7   |
+--------+------+---------+------------+-----------+----------+-----------+
Share of each weekday in each regime (%)
         Mon   Tue   Wed   Thu   Fri   Sat   Sun
regime                                          
0       32.0  34.0  34.0  32.0  28.0  23.0  21.0
1       68.0  66.0  66.0  68.0  72.0  77.0  79.0

k=3  level_only/kmeans/002_real_price_log_kmeans_k3
+--------+------+---------+------------+-----------+----------+-----------+
| Regime | Days | Share % | Real price | Reservoir | Mean ONI | El Nino % |
+--------+------+---------+------------+----------

Regime strip for every k, one row per k.

In [11]:
regime_strips(results)

alt.VConcatChart(...)